# CS229 Simplified PS3 — Unsupervised Learning

参考 CS229 K-means、PCA 与 semi-supervised EM 作业题型。

这不是 Stanford 官方作业副本，而是用于自学的**简化原创版本**。题型参考公开的
CS229 problem sets；公式以 2026 Main Notes 为准，数据由本仓库确定性生成。

- [Spring 2026 课程主页](https://cs229.stanford.edu/index.html-spr26)
- [Spring 2026 公开视频](https://www.youtube.com/playlist?list=PLaqpC4kq8Gpw)
- [CS229 2026 Main Notes](https://cs229.stanford.edu/main_notes.pdf)
- [Stanford 官方公开的 Summer 2020 problem sets](https://cs229.stanford.edu/summer2020/)
- [公开可见的 Summer 2025 作业结构参考](https://github.com/MDzimah/Stanford-University-CS229-Machine-Learning)

**使用规则：**先手推，再写代码。不要先看学生 solution。每道题完成后才把对应的
`RUN_*_CHECKS` 改成 `True`。检查不通过时，从 shape 和公式开始定位。

## Goal

对应 Spring 2026 Lecture 9–10。通过三个可见实验理解：K-means 如何压缩颜色、
PCA 如何寻找主要方向、EM 如何在隐变量存在时交替更新。

## Setup

数据和小图像都由构建脚本生成，不需要联网。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(229)

candidates = [Path("data"), Path("problem_sets/ps3_unsupervised/data")]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run: python scripts/build_problem_sets.py")

print("Data directory:", DATA_DIR.resolve())


In [ ]:
points = np.loadtxt(DATA_DIR / "points.csv", delimiter=",", skiprows=1)
image = np.load(DATA_DIR / "tiny_image.npy")
print("points:", points.shape, "image:", image.shape)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].scatter(points[:, 0], points[:, 1], s=12)
axes[0].set_title("Unlabelled points")
axes[1].imshow(image)
axes[1].set_title("Original tiny image")
axes[1].axis("off")
plt.show()


## Problem 1 — K-means 与图像压缩

**课堂连接：**Lecture 9。

把每个 RGB 像素当成三维样本。实现 centroid 初始化、assignment 和 centroid update，
然后用 4 种颜色重建图像。

**预期：**loss 在迭代中不增加；压缩图像 shape 与原图完全相同。

In [ ]:
def assign_clusters(X, centroids):
    # START TODO
    raise NotImplementedError("Return nearest-centroid index for every row")
    # END TODO

def update_centroids(X, assignment, k, old_centroids):
    # START TODO
    raise NotImplementedError("Return one mean vector per cluster; handle empty clusters")
    # END TODO

def fit_kmeans(X, k, rng, max_steps=50):
    # START TODO
    raise NotImplementedError("Initialize and alternate assignment/update")
    # END TODO


In [ ]:
RUN_P1_CHECKS = False
if RUN_P1_CHECKS:
    pixels = image.reshape(-1, 3)
    centroids, assignment, history = fit_kmeans(pixels, 4, rng)
    compressed = centroids[assignment].reshape(image.shape)
    print("loss history:", history)
    assert compressed.shape == image.shape
    assert np.all(np.diff(history) <= 1e-10)
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(image); axes[0].set_title("original")
    axes[1].imshow(np.clip(compressed, 0, 1)); axes[1].set_title("4 colors")
    for ax in axes: ax.axis("off")
    plt.show()
else:
    print("P1 checks are off.")


> **你的答案（请双击本格后填写）**
>
> TODO：解释为什么每次 assignment/update 都不会增加 K-means objective。

## Problem 2 — PCA 降到一维再重建

**课堂连接：**Lecture 10。

1. 对数据逐列中心化。
2. 计算 covariance matrix。
3. 用 `np.linalg.eigh` 找最大特征值对应方向。
4. 投影到一维，再重建回二维。

**预期：**主方向 shape 为 `(2, 1)`；重建误差小于“所有点都预测为均值”的基线。

In [ ]:
def fit_pca(X, n_components=1):
    # Return mean and principal directions sorted by decreasing eigenvalue.
    # START TODO
    raise NotImplementedError("Center, form covariance, and use np.linalg.eigh")
    # END TODO

def pca_transform(X, mean, components):
    # START TODO
    raise NotImplementedError("Project centered data")
    # END TODO

def pca_inverse_transform(Z, mean, components):
    # START TODO
    raise NotImplementedError("Reconstruct in original space")
    # END TODO


In [ ]:
RUN_P2_CHECKS = False
if RUN_P2_CHECKS:
    mean, components = fit_pca(points, n_components=1)
    Z = pca_transform(points, mean, components)
    reconstructed = pca_inverse_transform(Z, mean, components)
    error = np.mean((points - reconstructed) ** 2)
    baseline = np.mean((points - mean) ** 2)
    print("components:", components.shape, "Z:", Z.shape)
    print("reconstruction MSE:", error, "baseline:", baseline)
    assert components.shape == (2, 1)
    assert Z.shape == (len(points), 1)
    assert error < baseline
else:
    print("P2 checks are off.")


## Problem 3 — 一维两成分 GMM 的 EM（选做）

**课堂连接：**Lecture 9–10。

不调用 sklearn。实现 E-step 的 responsibility，以及 M-step 的 $\pi_k,\mu_k,\sigma_k^2$
更新。为避免下溢，可以在概率中加很小的 epsilon。

**预期：**每行 responsibility 之和为 1；log-likelihood 不应明显下降。

In [ ]:
x_gmm = np.concatenate([
    rng.normal(-2.0, 0.5, size=80),
    rng.normal(2.0, 0.8, size=120),
])

def gmm_e_step(x, weights, means, variances):
    # START TODO
    raise NotImplementedError("Return responsibilities with shape (m, k)")
    # END TODO

def gmm_m_step(x, responsibilities):
    # START TODO
    raise NotImplementedError("Return updated weights, means, and variances")
    # END TODO


In [ ]:
RUN_P3_CHECKS = False
if RUN_P3_CHECKS:
    weights = np.array([0.5, 0.5])
    means = np.array([-1.0, 1.0])
    variances = np.array([1.0, 1.0])
    for _ in range(30):
        responsibilities = gmm_e_step(x_gmm, weights, means, variances)
        assert np.allclose(responsibilities.sum(axis=1), 1.0)
        weights, means, variances = gmm_m_step(x_gmm, responsibilities)
    print("weights:", weights)
    print("means:", means)
    print("variances:", variances)
else:
    print("P3 checks are off. This problem is optional.")


## Checks

- [ ] K-means objective 不随迭代增加。
- [ ] PCA 的 components、投影和重建 shape 全部写对。
- [ ] 我能解释 K-means hard assignment 与 GMM soft responsibility 的差别。

## Next Steps

Spring 2026 的 diffusion、representation、LLM 和 RL 暂不伪装成官方 problem set。
后续只在找到可靠课堂案例或明确标注“原创 lecture lab”时再添加。